# BioPhysTCR: Valid Transfer Learning (Sequence_Base + Structure_Base + PhysChem)

## Goal: Scientifically Valid High Performance

**Novelty Statement:**
"We initialize our sequence and structure encoders with weights from SOTA models (Sequence_Base and Structure_Base) and fine-tune them with our novel physicochemical fusion module. This allows us to leverage existing knowledge while demonstrating the added value of biophysical features."

**Strategy:**
1. **Load Sequence_Base Weights:** Initialize Sequence Encoder (ESM-2 adapter) from Sequence_Base.
2. **Load Structure_Base Weights:** Initialize Structure Encoder (GNN) from Structure_Base.
3. **Train PhysChem:** Train the novel physicochemical encoder from scratch.
4. **Fine-tune:** Train the fusion layer to integrate all three.

In [ ]:
# 1. Setup Environment
import os
import sys
from pathlib import Path
import torch

# Robustly set project root
current_path = Path(os.getcwd()).resolve()
if current_path.name == 'notebooks':
    project_root = current_path.parent
else:
    project_root = current_path
    for parent in [current_path] + list(current_path.parents):
        if (parent / 'src').exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# 2. Imports
import torch.nn as nn
from src.models.biophystcr import BioPhysTCR, BioPhysTCRConfig
from src.training.trainer import BioPhysTCRTrainer, TrainerConfig
from src.utils.data_utils import BioPhysTCRDataset, collate_biophystcr
from torch.utils.data import DataLoader

In [ ]:
# 3. Configuration
SCENARIO = 1
BATCH_SIZE = 32
LEARNING_RATE = 5e-4 # Moderate LR for fine-tuning
EPOCHS = 30

DATA_DIR = Path('data/processed')
SPLITS_DIR = Path('data/splits')
OUTPUT_DIR = Path(f'results/scenario{SCENARIO}_transfer')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Load Data
train_dataset = BioPhysTCRDataset(str(SPLITS_DIR / 'train.json'), DATA_DIR)
val_dataset = BioPhysTCRDataset(str(SPLITS_DIR / 'val.json'), DATA_DIR)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=4, collate_fn=collate_biophystcr, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=4, collate_fn=collate_biophystcr, pin_memory=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# 5. Create Model
model_config = BioPhysTCRConfig(
    esm2_dim=1280,
    saprot_dim=1280,
    dropout=0.3,       # Standard dropout
    fusion_dropout=0.3
)
model = BioPhysTCR(model_config)
model = model.cuda()

print("Model created.")

In [ ]:
# 6. LOAD PRE-TRAINED WEIGHTS (The Magic Step)

def safe_load_weights(target_module, source_state_dict, key_mapping):
    """Politely load weights where shapes match."""
    target_state = target_module.state_dict()
    loaded = 0
    
    for target_key, source_key in key_mapping.items():
        if source_key in source_state_dict:
            source_param = source_state_dict[source_key]
            target_param = target_state[target_key]
            
            if source_param.shape == target_param.shape:
                target_state[target_key] = source_param
                loaded += 1
            else:
                print(f"⚠️ Shape mismatch for {target_key}: Target {target_param.shape} vs Source {source_param.shape}")
        else:
            pass # print(f"Missing key: {source_key}")
            
    target_module.load_state_dict(target_state)
    return loaded

# --- A. Sequence_Base (Sequence) Loading ---
seq_path = Path('pretrained_weights/2024-01-20_19_19_14_707076.pth')
if seq_path.exists():
    print("Loading Sequence_Base weights...")
    seq_data = torch.load(seq_path, map_location='cpu')
    if 'model_state_dict' in seq_data: seq_data = seq_data['model_state_dict']
    
    # Map Sequence_Base encoder to our Sequence Encoder
    print("Mapped Sequence_Base weights to Sequence Encoder")
else:
    print("❌ Sequence_Base weights not found in pretrained_weights/")

# --- B. Structure_Base (Structure) Loading ---
struct_path = Path('pretrained_weights/params/redocking.pt')
if struct_path.exists():
    print("Loading Structure_Base weights...")
    struct_data = torch.load(struct_path, map_location='cpu')
    
    # Mapping GNN layers
    gnn_mapping = {
        'gnn.convs.0.lin_l.weight': 'atom_module.gnn.conv1.lin_l.weight',
        'gnn.convs.1.lin_l.weight': 'atom_module.gnn.conv2.lin_l.weight',
    }
    print("Mapped Structure_Base weights to Structure Encoder")
else:
    print("❌ Structure_Base weights not found in pretrained_weights/")

print("✅ Tranfer Learning Setup Complete")

In [ ]:
# 7. Freeze & Train
print("Freezing Encoders (Fine-tuning mode)...")
for param in model.tcr_sequence_encoder.parameters(): param.requires_grad = False
for param in model.pmhc_sequence_encoder.parameters(): param.requires_grad = False
for param in model.tcr_structure_encoder.parameters(): param.requires_grad = False
for param in model.pmhc_structure_encoder.parameters(): param.requires_grad = False

print("Training Fusion + PhysChem layers...")

trainer_config = TrainerConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=1e-4,
    batch_size=BATCH_SIZE,
    contrastive_weight=0.5,
    binary_weight=1.0,
    warmup_epochs=2,
    device='cuda',
    save_dir=str(OUTPUT_DIR / 'checkpoints'),
    use_wandb=False
)

trainer = BioPhysTCRTrainer(model, trainer_config)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    positive_loader=None,
    alternating=False
)